In [ ]:
import os
import re
import json
import time
import tiktoken
import sqlite3

from IPython.core.display_functions import clear_output
from dotenv import load_dotenv
from pathlib import Path
from tuutrag.data import D
from tuutrag.types import *
from tuutrag.prompt import create_batch_request
from tuutrag.prompts.manager import load_prompt
from tuutrag.utils import count_batch_request_tokens
from openai import OpenAI
from tqdm import tqdm

In [ ]:
# %load_ext autoreload
# %autoreload 2

In [ ]:
load_dotenv()

OPENAI_MODEL: str = "gpt-4.1-mini"

openai_key: str | None = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise ValueError("OPENAI_API_KEY is not set in environment variables.")

openai_client: OpenAI = OpenAI(api_key=openai_key)

In [ ]:
# ── Load vector_sim_branches.json into list of lists ─────────────────────
VECTOR_SIM_PATH = "../data/processed/vector_sim_branches.json"

with open(VECTOR_SIM_PATH, "r") as f:
    raw_data: list[dict] = json.load(f)

print(f"Loaded {len(raw_data):,} records from {VECTOR_SIM_PATH}")

In [ ]:
# Build list of lists:
# [uuid, entities_str, artifact_summary, local_relations_str, [{"uuid": ..., "chunk": ...}, ...]]

all_branches: list[list] = []

for record in raw_data:
    uuid: str = record["uuid"]
    entities_str: str = " | ".join(record["entities"]) if record["entities"] else ""
    artifact_summary: str = record.get("artifact_summary", "")

    # Format local_relations as a readable string for the prompt
    local_relations_str: str = "\n".join(
        f"{r[0]}, {r[1]}, {r[2]}"
        for r in record.get("local_relations", [])
        if len(r) == 3
    )

    sim_branches: list[dict] = record.get("sim_branches", [])

    all_branches.append([
        uuid,
        entities_str,
        artifact_summary,
        local_relations_str,
        sim_branches,
    ])

print(f"Built {len(all_branches):,} branch records")
print(f"\nSample record:")
print(f"  UUID:              {all_branches[0][0]}")
print(f"  Entities (first 100 chars): {all_branches[0][1][:100]}...")
print(f"  Summary (first 100 chars):  {all_branches[0][2][:100]}...")
print(f"  Relations (first 100 chars): {all_branches[0][3][:100]}...")
print(f"  Sim branches count: {len(all_branches[0][4])}")

In [ ]:
# ── Deduplicate by UUID ──────────────────────────────────────────────────
seen: set[str] = set()
unique_branches: list[list] = []
for branch in all_branches:
    if branch[0] not in seen:
        seen.add(branch[0])
        unique_branches.append(branch)

print(f"Unique branches: {len(unique_branches):,}")

In [ ]:
SCHEME: str = "o200k_base"
enc: tiktoken.Encoding = tiktoken.get_encoding(SCHEME)

total_tokens: int = 0
for branch in unique_branches:
    # uuid[0], entities[1], artifact_summary[2], relations[3], sim_branches[4]
    total_tokens += len(enc.encode(branch[1]))  # entities
    total_tokens += len(enc.encode(branch[2]))  # artifact_summary
    total_tokens += len(enc.encode(branch[3]))  # local_relations
    for sb in branch[4]:                         # sim_branch chunks
        total_tokens += len(enc.encode(sb["chunk"]))

print(f"Estimated input tokens: {total_tokens:,}")

In [ ]:
# ── Build raw_chunk from all sim_branch chunks ───────────────────────────
# The prompt's raw_chunk = all chunks concatenated (current + similar branches)

all_requests: list[BatchRequest] = [
    create_batch_request(
        custom_id=branch[0],  # uuid
        model=OPENAI_MODEL,
        **load_prompt(
            "relation_global_uni.j2",
            raw_chunk="\n\n".join(sb["chunk"] for sb in branch[4]),
            entities=branch[1],          # joined entity string
            relations=branch[3],         # local relations string
            artifact_summary=branch[2],  # artifact summary
        )
    )
    for branch in tqdm(unique_branches, desc="Creating Batch Requests")
]

token_counts: list[int] = [
    count_batch_request_tokens(
        enc=enc,
        payload=req
    ) for req in tqdm(all_requests, desc="Counting Total Tokens")
]

In [ ]:
print(len(all_requests))
print(f"Total tokens across all requests: {sum(token_counts):,}")

In [ ]:
print(f"Average tokens per request: {sum(token_counts) / len(all_requests):,.2f}")
print(f"Average entity count: {sum(len(b[1].split(' | ')) for b in unique_branches) / len(unique_branches):,.2f}")
print(f"Average relations count: {sum(len(b[3].splitlines()) for b in unique_branches) / len(unique_branches):,.2f}")

In [ ]:
# ── Filter out already-processed UUIDs ───────────────────────────────────
master_file: Path = D.processed / "global_relations.json"

try:
    with open(master_file, "r", encoding="utf-8") as f:
        processed_ids: set[str] = {obj["uuid"] for obj in json.load(f)}
except (FileNotFoundError, json.JSONDecodeError):
    processed_ids = set()

pending_requests: list[BatchRequest] = [
    req for req in all_requests if req["custom_id"] not in processed_ids
]
pending_token_counts: list[int] = [
    token_counts[i] for i, req in enumerate(all_requests)
    if req["custom_id"] not in processed_ids
]

print(f"Total requests    : {len(all_requests):,}")
print(f"Already processed : {len(processed_ids):,}")
print(f"Pending           : {len(pending_requests):,}")

In [ ]:
all_requests = pending_requests
token_counts = pending_token_counts

In [ ]:
MAX_TOKENS_PER_BATCH: int = 1_400_000

batches: list[list[BatchRequest]] = []
current_batch: list[BatchRequest] = []
current_tokens: int = 0

for req, req_tokens in zip(all_requests, token_counts):
    if current_tokens + req_tokens > MAX_TOKENS_PER_BATCH and current_batch:
        batches.append(current_batch)
        current_batch = []
        current_tokens = 0
    current_batch.append(req)
    current_tokens += req_tokens

if current_batch:
    batches.append(current_batch)

print(f"Total requests : {len(all_requests):,}")
print(f"Total batches  : {len(batches):,}")
for i, batch in enumerate(batches):
    batch_tokens = sum(token_counts[all_requests.index(req)] for req in batch)
    print(f"  Batch {i + 1}: {len(batch):,} requests | {batch_tokens:,} tokens")

In [ ]:
print(f"First batch preview (showing up to 1 request):")
for req in batches[0][:1]:
    print(json.dumps(req, indent=2))

In [ ]:
POLL_INTERVAL: int = 30
RETRY_WAIT:    int = 120
MAX_RETRIES:   int = 5

metadata_path: Path = D.global_relation_batches / "batch_metadata.json"

In [ ]:
def _save_metadata(metadata: list[dict]) -> None:
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

def _get_metadata() -> list[dict]:
    with open(metadata_path, "r", encoding="utf-8") as f:
        return json.load(f)

def _remove_from_metadata(batch_id: str) -> None:
    metadata = _get_metadata()
    _save_metadata([m for m in metadata if m.get("id") != batch_id])

In [ ]:
def _append_branches(output_text: str) -> int:
    """Parse <|> delimited LLM output and append relations to master file. Returns count added."""
    responses: list[dict] = [
        json.loads(line)
        for line in output_text.strip().split("\n")
        if line.strip()
    ]

    try:
        with open(master_file, "r", encoding="utf-8") as f:
            existing: list[dict] = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        existing = []

    uuid_index: dict[str, int] = {
        entry["uuid"]: idx for idx, entry in enumerate(existing)
    }

    added: int = 0
    for response in responses:
        res_custom_id: str = response["custom_id"]
        res_content: str = response["response"]["body"]["choices"][0]["message"]["content"]

        relations: list[str] = [
            relation.strip()
            for relation in res_content.split("<|>")
            if relation.strip()
        ]

        if not relations:
            continue

        if res_custom_id in uuid_index:
            idx = uuid_index[res_custom_id]
            existing[idx]["relationships"].extend(relations)
        else:
            existing.append({
                "uuid": res_custom_id,
                "relationships": relations
            })
            uuid_index[res_custom_id] = len(existing) - 1

        added += len(relations)

    with open(master_file, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)

    return added

In [ ]:
def _resubmit(old_batch_id: str) -> str:
    """Resubmit a failed batch from its original .jsonl and update metadata."""
    metadata  = _get_metadata()
    entry     = next(m for m in metadata if m["id"] == old_batch_id)

    with open(D.global_relation_batches / entry["file_name"], "rb") as f:
        batch_file = openai_client.files.create(file=f, purpose="batch")

    new_job = openai_client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

    entry["id"] = new_job.id
    _save_metadata(metadata)

    return new_job.id

In [ ]:
# ── Cleanup: delete old batch files and cancel old jobs ──────────────────
pattern = re.compile(r"^global_relations_batch_\d+\.jsonl$")

deleted_local = [
    f for f in metadata_path.parent.iterdir()
    if f.is_file() and pattern.match(f.name) and not f.unlink()
]
print(f"Deleted locally  : {len(deleted_local):,} .jsonl files")

deleted_files  = 0
cancelled_jobs = 0

try:
    metadata = _get_metadata()
    for entry in metadata:
        file_id = entry.get("input_file_id")
        if file_id:
            try:
                openai_client.files.delete(file_id)
                deleted_files += 1
                print(f"  Deleted file      : {file_id}  ({entry['file_name']})")
            except Exception as e:
                print(f"  Could not delete file {file_id}: {e}")

        batch_id = entry.get("id")
        if batch_id:
            try:
                openai_client.batches.cancel(batch_id)
                cancelled_jobs += 1
                print(f"  Cancelled batch   : {batch_id}")
            except Exception as e:
                print(f"  Could not cancel batch {batch_id}: {e}")

except FileNotFoundError:
    print("  No batch_metadata.json found — skipping remote cleanup.")

print(f"Deleted files    : {deleted_files:,}")
print(f"Cancelled jobs   : {cancelled_jobs:,}")

if metadata_path.exists():
    metadata_path.unlink()
    print("Deleted          : batch_metadata.json")

In [ ]:
# ── Upload batch files ───────────────────────────────────────────────────
batch_file_ids: list[dict] = []

for idx, batch in enumerate(tqdm(batches, desc="Uploading batch files")):
    file_name: str = f"global_relations_batch_{idx}.jsonl"

    with open(metadata_path.parent / file_name, "w", encoding="utf-8") as f:
        for request in batch:
            f.write(json.dumps(request) + "\n")

    with open(metadata_path.parent / file_name, "rb") as f:
        batch_file = openai_client.files.create(file=f, purpose="batch")

    batch_file_ids.append({
        "file_name": file_name,
        "input_file_id": batch_file.id,
    })

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(batch_file_ids, f, indent=2)

print(f"\nUploaded {len(batch_file_ids):,} files. Ready to submit jobs.")

In [ ]:
# ── Submit and monitor batches ───────────────────────────────────────────
print(f"Submitting and processing {len(batch_file_ids)} batch(es) one at a time.\n")

for file_entry in batch_file_ids:
    file_name: str = file_entry["file_name"]
    input_file_id: str = file_entry["input_file_id"]
    retries: int = 0

    batch_job = openai_client.batches.create(
        input_file_id=input_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    batch_id: str = batch_job.id

    metadata = _get_metadata()
    for m in metadata:
        if m["input_file_id"] == input_file_id:
            m.update({"id": batch_id, **batch_job.model_dump()})
    _save_metadata(metadata)

    print(f"{file_name} | submitted: {batch_id}")

    while True:
        job = openai_client.batches.retrieve(batch_id)
        status = job.status

        if status == "completed":
            output_resp = openai_client.files.content(job.output_file_id)
            added = _append_branches(output_resp.text)
            _remove_from_metadata(batch_id)
            print(f"Complete - {added} relations added | {len(_get_metadata())} remaining\n")
            break

        elif status == "failed":
            errors = job.errors.data if job.errors else []
            token_limit = any(
                "enqueued token limit" in (e.message or "").lower()
                for e in errors
            )
            if token_limit and retries < MAX_RETRIES:
                retries += 1
                print(f"Token limit hit - waiting {RETRY_WAIT}s then resubmitting (attempt {retries}/{MAX_RETRIES})")
                time.sleep(RETRY_WAIT)
                batch_id = _resubmit(batch_id)
                print(f"Resubmitted as {batch_id}")
            elif retries >= MAX_RETRIES:
                print(f"Max retries ({MAX_RETRIES}) reached - skipping.\n")
                _remove_from_metadata(batch_id)
                break
            else:
                messages = [e.message for e in errors]
                print(f"Failed: {messages} - skipping.\n")
                _remove_from_metadata(batch_id)
                break

        elif status in ("cancelling", "cancelled", "expired"):
            print(f"Status '{status}' - skipping.\n")
            _remove_from_metadata(batch_id)
            break

        else:
            completed = job.request_counts.completed
            total = job.request_counts.total
            clear_output(wait=True)
            print(f"{file_name} | {batch_id}")
            print(f"[{status}] {completed}/{total} - next check in {POLL_INTERVAL}s...")
            time.sleep(POLL_INTERVAL)

print("All batches processed.")

In [ ]:
# ── Count results ────────────────────────────────────────────────────────
def count_relations():
    with open(master_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    total_relations: int = sum(len(entry["relationships"]) for entry in data)
    print(f"Total relationships extracted: {total_relations:,}")

def count_unique_relations():
    with open(master_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    unique_relations: set[str] = set()
    for entry in data:
        unique_relations.update(entry["relationships"])
    print(f"Unique relationships extracted: {len(unique_relations):,}")

In [ ]:
count_relations()
count_unique_relations()

In [ ]:
# ── Recovery function ────────────────────────────────────────────────────
def recover_batch_results(
    openai_client: OpenAI,
    output_file: Path,
    batch_prefix: str = "global_relations_batch_",
    batch_range: range = range(0, 51),
) -> int:
    target_files: set[str] = {
        f"{batch_prefix}{i}.jsonl" for i in batch_range
    }
    print(f"Looking for {len(target_files):,} batch files ({batch_prefix}0..{batch_range[-1]})\n")

    file_name_to_id: dict[str, str] = {}
    after: str | None = None

    while True:
        kwargs: dict = {"purpose": "batch", "limit": 100}
        if after:
            kwargs["after"] = after
        page = openai_client.files.list(**kwargs)
        files = page.data
        if not files:
            break
        for f in files:
            if f.filename in target_files:
                file_name_to_id[f.filename] = f.id
        after = files[-1].id
        if not page.has_more:
            break

    print(f"Matched {len(file_name_to_id):,} input files on OpenAI\n")

    target_input_ids: set[str] = set(file_name_to_id.values())
    id_to_name: dict[str, str] = {v: k for k, v in file_name_to_id.items()}

    matched_batches: list[tuple] = []
    after = None
    total_scanned: int = 0

    while True:
        kwargs = {"limit": 100}
        if after:
            kwargs["after"] = after
        page = openai_client.batches.list(**kwargs)
        batch_list = page.data
        if not batch_list:
            break
        for job in batch_list:
            total_scanned += 1
            if job.input_file_id in target_input_ids:
                file_name = id_to_name[job.input_file_id]
                matched_batches.append((job, file_name))
        after = batch_list[-1].id
        if not page.has_more:
            break

    print(f"Scanned {total_scanned:,} batches — matched {len(matched_batches):,}\n")

    if output_file.exists():
        with open(output_file, "r", encoding="utf-8") as f:
            try:
                existing: list[dict] = json.load(f)
            except json.JSONDecodeError:
                existing = []
    else:
        existing = []

    uuid_index: dict[str, int] = {
        entry["uuid"]: idx for idx, entry in enumerate(existing)
    }

    recovered: int = 0
    retrieved: int = 0
    skipped: int = 0
    failed: int = 0

    for job, file_name in matched_batches:
        batch_id: str = job.id
        if job.status != "completed":
            skipped += 1
            print(f"  [{job.status:>12}] {file_name} ({batch_id}) — skipping")
            continue
        if not job.output_file_id:
            skipped += 1
            print(f"  [no output   ] {file_name} ({batch_id}) — skipping")
            continue
        try:
            output_resp = openai_client.files.content(job.output_file_id)
            output_text: str = output_resp.text
        except Exception as e:
            failed += 1
            print(f"  [download err] {file_name} ({batch_id}): {e}")
            continue

        retrieved += 1
        print(f"  ✓ Retrieved {file_name} — batch {batch_id}")

        responses: list[dict] = [
            json.loads(line)
            for line in output_text.strip().split("\n")
            if line.strip()
        ]

        for response in responses:
            res_custom_id: str = response["custom_id"]
            res_content: str = (
                response["response"]["body"]["choices"][0]["message"]["content"]
            )
            if not res_content.strip():
                continue
            relations: list[str] = [
                f"<|>{segment.strip()}<|>"
                for segment in res_content.split("<|>")
                if segment.strip()
            ]
            if not relations:
                continue
            if res_custom_id in uuid_index:
                idx = uuid_index[res_custom_id]
                existing[idx]["relationships"].extend(relations)
            else:
                existing.append({
                    "uuid": res_custom_id,
                    "relationships": relations,
                })
                uuid_index[res_custom_id] = len(existing) - 1
            recovered += len(relations)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)

    print(f"\nTarget batch files       : {len(target_files):,}")
    print(f"Input files found        : {len(file_name_to_id):,}")
    print(f"Batches matched          : {len(matched_batches):,}")
    print(f"Completed & retrieved    : {retrieved:,}")
    print(f"Skipped (not completed)  : {skipped:,}")
    print(f"Failed (404 / errors)    : {failed:,}")
    print(f"Relationships recovered  : {recovered:,}")
    print(f"Saved to                 : {output_file}")

    return recovered

In [ ]:
# ── Fix/parse global relations ───────────────────────────────────────────
def parse_relation(raw: str) -> list[str] | None:
    line = raw.strip().strip('"')
    line = re.sub(r"^<\|?>", "", line)
    line = re.sub(r"<\|?>$", "", line)
    line = line.strip()
    if not line:
        return None
    first = line.find(",")
    if first == -1:
        return None
    second = line.find(",", first + 1)
    if second == -1:
        return None
    source   = line[:first].strip()
    relation = line[first + 1 : second].strip()
    target   = line[second + 1 :].strip()
    if not source or not relation or not target:
        return None
    return [source, relation, target]


def fix_relations_json(
    input_path: str | Path,
    output_path: str | Path,
) -> dict[str, int]:
    input_path  = Path(input_path)
    output_path = Path(output_path)
    with open(input_path, "r", encoding="utf-8") as f:
        data: list[dict] = json.load(f)

    total_raw: int = 0
    total_parsed: int = 0
    total_skipped: int = 0
    skipped_lines: list[dict] = []
    cleaned: list[dict] = []

    for entry in data:
        uuid = entry["uuid"]
        raw_rels = entry.get("relationships", [])
        parsed: list[list[str]] = []
        for rel_str in raw_rels:
            total_raw += 1
            triple = parse_relation(rel_str)
            if triple:
                parsed.append(triple)
                total_parsed += 1
            else:
                total_skipped += 1
                skipped_lines.append({"uuid": uuid, "raw": rel_str})
        cleaned.append({"uuid": uuid, "relationships": parsed})

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, indent=2, ensure_ascii=False)

    summary = {
        "input_relationships": total_raw,
        "output_triples": total_parsed,
        "skipped": total_skipped,
    }
    print(f"Input relationships  : {total_raw:,}")
    print(f"Output triples       : {total_parsed:,}")
    print(f"Skipped (unparseable): {total_skipped:,}")
    print(f"Saved to             : {output_path}")

    if skipped_lines:
        skipped_path = output_path.with_name(output_path.stem + "_skipped.json")
        with open(skipped_path, "w", encoding="utf-8") as f:
            json.dump(skipped_lines, f, indent=2, ensure_ascii=False)
        print(f"Skipped lines saved  : {skipped_path}")

    return summary

In [ ]:
summary = fix_relations_json(
    input_path=master_file,
    output_path=D.processed / "global_relations_fixed.json",
)

In [ ]:
def count_usable_tuples(relations_path: str | Path) -> int:
    with open(relations_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    usable: int = 0
    for entry in data:
        for triple in entry.get("relationships", []):
            if all(triple):
                usable += 1
    print(f"Usable triples (non-empty source/relation/target): {usable:,}")
    return usable

In [ ]:
count_usable_tuples(D.processed / "global_relations.json")